In [1]:
import pandas as pd
from collections import Counter
from pathlib import Path


In [2]:
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ,.-"
ALLOWED = set(ALPHABET)

def clean_text(text: str) -> str:
    """
    Convert to uppercase and keep only characters
    in the assignment alphabet.
    """
    text = text.upper()
    return "".join(ch for ch in text if ch in ALLOWED)


In [3]:
raw_text = Path("0.txt").read_text(encoding="utf-8", errors="ignore")
ciphertext = clean_text(raw_text)

len(ciphertext)


972

In [4]:
def one_gram_table(text: str) -> pd.DataFrame:
    counter = Counter(text)
    total = len(text)

    rows = []
    for ch in ALPHABET:
        count = counter.get(ch, 0)
        percent = (count / total) * 100 if total else 0
        rows.append([ch, count, round(percent, 2)])

    df = pd.DataFrame(rows, columns=["Symbol", "Count", "Percentage (%)"])
    return df

one_gram_df = one_gram_table(ciphertext)
one_gram_df


,Symbol,Count,Percentage (%)
0,A,23,2.37
1,B,26,2.67
2,C,45,4.63
3,D,25,2.57
4,E,33,3.40
5,F,42,4.32
6,G,48,4.94
7,H,21,2.16
8,I,21,2.16
9,J,32,3.29


In [5]:
def ngram_table(text: str, n: int, top: int) -> pd.DataFrame:
    ngrams = [text[i:i+n] for i in range(len(text) - n + 1)]
    counter = Counter(ngrams)
    total = sum(counter.values())

    rows = []
    for gram, count in counter.most_common(top):
        percent = (count / total) * 100 if total else 0
        rows.append([gram, count, round(percent, 2)])

    df = pd.DataFrame(rows, columns=[f"{n}-gram", "Count", "Percentage (%)"])
    return df

digram_df = ngram_table(ciphertext, n=2, top=16)
digram_df


,2-gram,Count,Percentage (%)
0,UG,14,1.44
1,WC,14,1.44
2,SO,11,1.13
3,YF,11,1.13
4,ML,10,1.03
5,QO,9,0.93
6,",P",9,0.93
7,OP,8,0.82
8,PK,8,0.82
9,.X,8,0.82


In [6]:
trigram_df = ngram_table(ciphertext, n=3, top=11)
trigram_df


,3-gram,Count,Percentage (%)
0,UGS,4,0.41
1,GSO,4,0.41
2,FOP,4,0.41
3,OUG,3,0.31
4,WCM,3,0.31
5,UG.,3,0.31
6,G.J,3,0.31
7,VPU,3,0.31
8,RKY,3,0.31
9,KYF,3,0.31


In [7]:
from collections import Counter

with open("0.txt") as f:
    text = f.read()

digrams = [text[i:i+2] for i in range(len(text)-1)]
counter = Counter(digrams)

for d, c in counter.most_common(10):
    print(d, c)


UG 14
WC 14
SO 11
YF 11
ML 10
QO 9
,P 9
OP 8
PK 8
.X 8


In [ ]:
alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ,.-"
def num(c): 
    return alphabet.index(c)

TH = [num('T'), num('H')]
HE = [num('H'), num('E')]

UG = [num('U'), num('G')]
WC = [num('W'), num('C')]


In [ ]:
import numpy as np

P = np.array([TH, HE]).T     
C = np.array([UG, WC]).T      

mod = 29

det = int(round(np.linalg.det(P))) % mod
det_inv = pow(det, -1, mod)

P_inv = det_inv * np.array([[P[1,1], -P[0,1]],
                            [-P[1,0], P[0,0]]]) % mod

K = (C @ P_inv) % mod
print(K)


[[ 8  6]
 [24  2]]


In [10]:
key = "".join(alphabet[int(x)] for x in K.flatten())
print(key)


IGYC


In [15]:
K_inv = pow(int(round(np.linalg.det(K))), -1, mod) * \
        np.array([[K[1,1], -K[0,1]],
                  [-K[1,0], K[0,0]]]) % mod

plaintext = ""
for i in range(0, len(text)-1, 2):
    c = np.array([num(text[i]), num(text[i+1])])
    p = (K_inv @ c) % mod
    plaintext += alphabet[p[0]] + alphabet[p[1]]

print(plaintext[:90000000])


WEREFIXEDSOSTEADILYUPONABALLONTHERAILOFTHESHIPTHATSHEWOULDHAVEBEENSTARTLEDANDANNOYEDIFANYTHINGHADCHANCEDTOOBSCUREITFORASECOND.SHEHADBEGUNHERMEDITATIONSWITHASHOUTOFLAUGHTER,CAUSEDBYTHEFOLLOWINGTRANSLATIONFROMTRISTANINSHRINKINGTREPIDATIONHISSHAMEHESEEMSTOHIDEWHILETOTHEKINGHISRELATIONHEBRINGSTHECORPSE-LIKEBRIDE.SEEMSITSOSENSELESSWHATISAYSHECRIEDTHATITDID,ANDTHREWDOWNTHEBOOK.NEXTSHEHADPICKEDUPCOWPERSLETTERS,THECLASSICPRESCRIBEDBYHERFATHERWHICHHADBOREDHER,SOTHATONESENTENCECHANCINGTOSAYSOMETHINGABOUTTHESMELLOFBROOMINHISGARDEN,SHEHADTHEREUPONSEENTHELITTLEHALLATRICHMONDLADENWITHFLOWERSONTHEDAYOFHERMOTHERSFUNERAL,SMELLINGSOSTRONGTHATNOWANYFLOWER-SCENTBROUGHTBACKTHESICKLYHORRIBLESENSATIONANDSOFROMONESCENESHEPASSED,HALF-HEARING,HALF-SEEING,TOANOTHER.SHESAWHERAUNTLUCYARRANGINGFLOWERSINTHEDRAWING-ROOM.AUNTLUCY,SHEVOLUNTEERED,IDONTLIKETHESMELLOFBROOMITREMINDSMEOFFUNERALS.NONSENSE,RACHEL,AUNTLUCYREPLIEDDONTSAYSUCHFOOLISHTHINGS,DEAR.IALWAYSTHINKITAPARTICULARLYCHEERFULPLANT.
